# ReadtheDocs Retrieval Augmented Generation (RAG) using Milvus Lite

在本笔记本中，我们将使用 Milvus 文档页面来创建一个关于我们产品的聊天机器人。该聊天机器人将遵循 RAG（检索增强生成）流程：通过语义向量搜索获取数据片段，然后将问题与上下文作为提示输入大语言模型，以生成回答。

许多 RAG 演示使用 OpenAI 的嵌入模型和 DeepSeek 生成式 AI 模型。**而在本笔记本中，我们将演示一个完全开源的 RAG 系统栈。**

使用开源的问答系统并结合检索可以节省成本，因为我们几乎每次都能免费调用自有数据进行检索、评估和开发迭代。仅在最终生成对话时，才向 DeepSeek 发起一次付费调用。

In [1]:
import os

os.environ['NLTK_DATA'] = r'F:\Teewon\Milvue\model\nltk_data'

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [2]:
import sys,time
import numpy as np
from pprint import pprint

sys.path.append("..")
import milvus_utilities as _utils

## Start up a Milvus Lite

In [3]:
from pymilvus import MilvusClient
mc=MilvusClient('../milvus_agent.db')

collections=mc.list_collections()
print("Existing collections:",collections)

Existing collections: ['wikipedia']


## Load the Embedding Model checkpoint and use it to create vector embeddings

**嵌入模型**：我们将使用 HuggingFace 上提供的开源句子转换模型来编码文档文本。我们从 HuggingFace 下载该模型，并在本地运行。

以下两个模型参数值得注意：

1. EMBEDDING_DIM 指的是嵌入向量的维度或长度。在此情况下，输入文本中每个标记生成的嵌入向量长度相同，均为 1024。这种嵌入大小通常与基于 BERT 的模型相关，其嵌入被用于下游任务，如分类、问答或文本生成。

2. MAX_SEQ_LENGTH 是编码器模型可处理的最大输入序列长度。在这种情况下，如果输入序列超过 512 个标记，所有过长的部分将被（静默地！）截断。因此，需要采用分块策略，将输入文本分割成适合模型输入长度的若干小段。

In [4]:
import torch
from torch.nn import functional as F
from sentence_transformers import SentenceTransformer

# Initialize torch settings
torch.backends.cudnn.deterministic=True
DEVECE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device:{DEVECE}")

# Load the model form huggingface model hub
model_name="WhereIsAI/UAE-Large-V1"
encoder=SentenceTransformer(model_name,device='cuda')
print(type(encoder))
print(encoder)

# Get the model parameters and save for later
EMBEDDING_DIM=encoder.get_embedding_dimension()
MAX_SEQ_LENGTH_IN_TOKEN=encoder.get_max_seq_length()
MAX_SEQ_LENGTH=MAX_SEQ_LENGTH_IN_TOKEN
HF_EOS_TOKEN_LENGTH=1

# Inspect model parameters.
print(f"model_name: {model_name}")
print(f"EMBEDDING_DIM: {EMBEDDING_DIM}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")

device:cuda
<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
)
model_name: WhereIsAI/UAE-Large-V1
EMBEDDING_DIM: 1024
MAX_SEQ_LENGTH: 512


# Create a Milvus collection
在 Milvus 中，你可以将集合类比为 SQL 数据库中的“表”。该集合将包含以下内容：

- **模型架构**（或无架构的 Milvus 客户端）

    💡 你需要从嵌入模型中获取向量的 EMBEDDING_DIM 参数。常见取值如下：
    - sbert 嵌入模型：1024
    - ada-002 OpenAI 嵌入模型：1536

- **向量索引**，用于高效向量搜索
- **向量距离度量**，用于计算最近邻向量
- **一致性级别**：Milvus 支持事务一致性，但根据 CAP 定理，必须牺牲一定的延迟。💡 由于电影评论搜索并非关键任务，因此`eventually`在此处是可接受的。

# Add a Vector Index

向量索引用于确定在用户提交查询时，查找数据中与该查询最接近的向量所采用的向量搜索算法。

大多数向量索引根据数据库的使用场景（插入向量或搜索向量）而采用不同的参数组合：

- **插入向量**（创建模式）
- **搜索向量**（搜索模式）

请向下滚动文档页面，查看 Milvus 提供的不同向量索引列表。例如：

- FLAT — 确定性穷尽搜索
- IVF_FLAT 或 IVF_SQ8 — 哈希索引（随机近似搜索）
- HNSW — 图形索引（随机近似搜索）
- AUTOINDEX — 根据 OSS 与 Zilliz 云、GPU 类型及数据规模自动确定

除了搜索算法外，我们还需要指定**距离度量**，即定义向量空间中“接近”的标准。在下方单元格中选择了 `HNSW` 搜索索引。其可用的距离度量包括：

- L2 — L2 范数
- IP — 点积
- COSINE — 角度距离

💡 大多数应用场景更适合使用归一化嵌入（normalized embeddings），此时 L2 不适用（每个向量长度为1），IP 和 COSINE 相等。仅当您计划保持嵌入未归一化时，才应选择 L2。

In [5]:
COLLECTION_NAME="wikipedia"

M=16
efConstruction=M*2
INDEX_PARAMS={
    'M':M,
    'efConstruction':efConstruction,
}
index_params={
    'index_type':"HNSW",
    'metric_type':"COSINE",
    'params':INDEX_PARAMS,
}

has=mc.has_collection(COLLECTION_NAME)
if has:
    drop_result=mc.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection: {COLLECTION_NAME}")

mc.create_collection(
    COLLECTION_NAME,
    EMBEDDING_DIM,
    consistency_level='Eventually',
    auto_id=True,
    params=index_params,
)

print(f"Successfully created collection: {COLLECTION_NAME}")

Successfully dropped collection: wikipedia
Successfully created collection: wikipedia


# Insert data into Milvus

对于每个原始文本片段，我们将把四元组（`vector, text, source, h1, h2`）写入数据库。

**Milvus 客户端封装器仅能处理从字典列表中加载数据。**

否则，Milvus 通常支持从以下格式加载数据：

- pandas 数据框
- 字典列表

下面我们将使用 HuggingFace 提供的嵌入模型，下载其检查点，并在本地运行以作为编码器。

In [6]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader=WebBaseLoader("https://en.wikipedia.org/wiki/New_York_City")
docs=loader.load()

text_splitter=RecursiveCharacterTextSplitter(chunk_size=512,chunk_overlap=50)
print(f"Num docs:{len(docs)}")
chunks=text_splitter.split_documents(docs)
print(f"Num chunks:{len(chunks)}")

chunk_list=[]
for chunk in chunks:
    embeddings=torch.tensor(encoder.encode([chunk.page_content]))
    embeddings=np.array(embeddings/np.linalg.norm(embeddings))
    converted_values=list(map(np.float32,embeddings))[0]

    chunk_dict={
        'vector':converted_values,
        'chunk':chunk.page_content,
        'source':chunk.metadata['source'],
        'h1':chunk.metadata['title'][:50],
    }
    chunk_list.append(chunk_dict)

# Insert data into the Milvus collection.
print("Start inserting entities")
start_time = time.time()
insert_result = mc.insert(
    COLLECTION_NAME,
    data=chunk_list,
    append=True,
    progress_bar=True)
end_time = time.time()
print(f"Milvus Client insert time for {len(chunk_list)} vectors: {end_time - start_time} seconds")
# Milvus Client insert time for 646 vectors: 4.732278823852539 seconds

# After final entity is inserted, call flush to stop growing segments left in memory.
mc.flush(COLLECTION_NAME)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_10040\3234376542.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Num docs:1
Num chunks:778


C:\Users\Administrator\AppData\Local\Temp\ipykernel_10040\3234376542.py:15: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  embeddings=np.array(embeddings/np.linalg.norm(embeddings))


Start inserting entities
Milvus Client insert time for 778 vectors: 0.2050013542175293 seconds


## Define Evaluation Metrics

In [7]:
import types
_vmod=types.ModuleType("langchain_community.chat_models.vertexai")
class ChatVertexAI:
    pass
_vmod.ChatVertexAI=ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = _vmod
import langchain_community.llms as _ll
if not hasattr(_ll,'VertexAI'):
    class VertexAI:
        pass
    _ll.VertexAI=VertexAI

In [8]:
import dotenv
dotenv.load_dotenv('../.env')

from langchain_deepseek import ChatDeepSeek
from ragas.llms.base import LangchainLLMWrapper

llm_langchain=ChatDeepSeek(
    model='deepseek-v4-flash',
    temperature=0,
    api_key=os.getenv('DEEPSEEK_API_KEY'),
)

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

lc_embeddings=HuggingFaceEmbeddings(model_name='WhereIsAI/UAE-Large-V1')
emb_wrapper=LangchainEmbeddingsWrapper(embeddings=lc_embeddings)

# ============ 指标 ============
from ragas.metrics import (
    _context_recall,
    _context_precision,
    _faithfulness,
    _answer_similarity,
    _answer_relevancy,
    _answer_correctness
)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_10040\1799546822.py:5: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  emb_wrapper=LangchainEmbeddingsWrapper(embeddings=lc_embeddings)


In [10]:
from datasets import Dataset
from ragas import evaluate
def assemble_ragas_dataset(input_df, answer_col_name="Custom_RAG_answer",context_exists=True,row_number=-9999):
    subset_df=input_df.iloc[row_number:row_number+1,:]if row_number>=0 else input_df.copy()
    question_list=subset_df.Question.to_list()
    answer_list=subset_df[answer_col_name].to_list()

    if context_exists:
        context_list=subset_df.Custom_RAG_context.to_list()
        context_list=[[c] for c in context_list]
    else:
        context_list=[[" "] for _ in question_list]

    truth_list=subset_df.ground_truth_answer.to_list()
    truth_list=[[t] for t in truth_list]

    return Dataset.from_dict({
        "question":question_list,
        "context":context_list,
        "answer":answer_list,
        "ground_truth":truth_list,
    })

def evaluate_ragas(input_df,answer_col_name="Custom_RAG_answer",context_exists=True,row_number=-9999,metrics='final_only',llm=None,embeddings=None):
    ragas_input_ds=assemble_ragas_dataset(input_df,answer_col_name=answer_col_name,context_exists=context_exists,row_number=row_number)
    final_metrics=[_answer_similarity,_answer_relevancy,_answer_correctness]
    all_metrics=[_context_recall,_context_precision,_answer_relevancy,_faithfulness,_answer_similarity,_answer_correctness]

    return evaluate(
        ragas_input_ds,
        metrics=all_metrics if metrics!='final_only' else final_metrics,
        llm=llm,
        embeddings=embeddings
    )

In [11]:
import pandas as pd
from IPython.display import display

eval_df=pd.read_csv("../Evaluation/data/blog_eval_answers.csv",header=0,skip_blank_lines=True)
display(eval_df.head())

# Get all the questions.
question_list=eval_df.Question.to_list()

# Get all the ground truth answers.
truth_list = eval_df.ground_truth_answer.to_list()

# Get all the ground truth sources.
uri_list = eval_df.Uri.to_list()

# Get all the OpenAI Answers.
openai_answer_list = eval_df.OpenAI_RAG_answer.to_list()

,Question,ground_truth_answer,recursive_context_512_k_2,html_context_512_k_2,parent_context_1536_k1,semantic_context_k_1,semantic_context_k_2_summary,parent_context_1536_k1_text-embedding-3-small,Custom_RAG_answer,llama3_ollama_answer,...,llama3_groq_answer,mixtral_8x7b_anyscale_answer,OpenAI_RAG_answer,Custom_RAG_answer.1,Custom_RAG_context,Uri,H1,H2,Score,Reason
0,What do the parameters for HNSW mean?,"* M: maximum degree, or number of connections ...","node closest to the target in this layer, and ...","layer, finds the node closest to the target in...",Parameter Description Range Default value npro...,Index building parameters Parameter Descriptio...,* `M`: The maximum number of outgoing connecti...,Parameter Description Range Default value npro...,The parameters for HNSW (Hierarchical Navigabl...,* `M`: Maximum number of outgoing connections ...,...,* M: Maximum number of outgoing connections in...,"The parameters for HNSW, a graph-based indexin...",The HNSW parameters include the “nlist” which ...,The parameters for HNSW have the following mea...,"performance, HNSW limits the maximum degree of...",https://pymilvus.readthedocs.io/en/latest/para...,Index,Milvus support to create index to accelerate v...,NaN,NaN
1,What are good default values for HNSW paramete...,"M=16, efConstruction=32, ?ef=32",Select your Milvus distribution first. Index b...,you can set the top-K up to 8192 for any searc...,Select your Milvus distribution first. Index b...,Index building parameters Parameter Descriptio...,* `nlist`: This parameter controls the number ...,Parameter Description Range Default value npro...,* M=48 ?* efConstruction=200,"* For `nprobe`, a reasonable default value is ...",...,* nprobe: 16 ?* reorder_k: 128 ?* M: 128 ?* ef...,NaN,The default HNSW parameters for data size of 2...,For a data size of 25K vectors with a dimensio...,Metrics. Vector Index露 FLAT IVF_FLAT IVF_SQ8 I...,https://pymilvus.readthedocs.io/en/latest/para...,NaN,NaN,NaN,NaN
2,What does nlist vs nprobe mean in ivf_flat?,# nlist: controls how the vector data is part...,FAQ What is the difference between FLAT index ...,performance can be improved with minimal impac...,FAQ What is the difference between FLAT index ...,See Supported Metrics. IVF_FLAT IVF_FLAT divid...,**nlist (Number of List)**:\nThis parameter de...,1] FAQ What is the difference between FLAT ind...,- nlist refers to the number of clusters into ...,- `nlist` refers to the number of clusters (or...,...,- `nlist` refers to the number of clusters to ...,- `nlist` refers to the number of clusters in ...,The default distance metric used in AUTOINDEX ...,The default distance metric used in AUTOINDEX ...,The attributes of collection can be extracted ...,https://pymilvus.readthedocs.io/en/latest/tuto...,NaN,NaN,NaN,NaN
3,What is the default AUTOINDEX index and vector...,Index type = HNSW and distance metric=IP Inner...,"True, and auto_id is enabled for the primary k...","is set to True, and auto_id is enabled for the...","vector in the data to be inserted, are treated...","For a detailed explanation of the schema, refe...","According to the provided text, when creating ...","vector in the data to be inserted, are treated...",The default AUTOINDEX index in Milvus is IVF_S...,The default `AUTOINDEX` index type uses L2 (Eu...,...,The default distance metric for vector fields ...,The default distance metric for vector fields ...,"I'm sorry, but I couldn't find any information...",New York City was originally named New Amsterd...,Etymology\nSee also: Nicknames of New York Cit...,https://en.wikipedia.org/wiki/New_York_City,NaN,NaN,NaN,NaN


## Define a Custom Execution Loop for RAG

In [12]:
import requests,json

TOP_K=3

def milvus_lite_collection_search(question,top_k=3,filter_expr=None):
    """在本地 Milvus Lite 中检索与问题最相关的文档片段。"""
    q_vec=encoder.encode([question])
    q_vec=np.array(q_vec/np.linalg.norm(q_vec)) # L2 归一化
    query_vector=list(map(np.float32,q_vec))[0]

    results=mc.search(
        collection_name=COLLECTION_NAME,
        data=[query_vector],
        limit=top_k,
        output_fields=["chunk","source","h1"],
        filter=filter_expr,
    )

    hits=results[0]
    contexts=[hit['entity']['chunk'] for hit in hits]
    return  contexts,hits

In [ ]:
def get_deepseek_chat(user_prompt,retrieval_context,retrieval_source,message_history,temperature=0,frequency_penalty=2):
    """
    用 deepseek-v4-flash 生成回答，返回 (response_df, token_use_df)。
    说明：llm_langchain 是前面定义的 ChatDeepSeek 实例。
    """
    system_message=f"""
    Use the Context to answer the user's question. Be clear, factual, complete, concise.
    If the answer is not in the Context, say "I don't know".  Otherwise answer using this format:
    Context: {retrieval_context}
    Answer: The answer to the question.
    Grounding source: {retrieval_source}
    """
    messages=[
        {'role':"system",'content':system_message},
        {'role':"user","content":f"{user_prompt}"},
        {"role":"assistant","content":f"Relevant context:\n{retrieval_context}"},
    ]

    response=llm_langchain.invoke(
        message_history+messages,
    )
    message_history=message_history+messages[1:]

    usage=getattr(response,"usage_metadata",None) or {}
    token_dict={
        'prompt_tokens':usage.get('input_tokens',0),
        'completion_tokens':usage.get('output_tokens',0),
        'total_tokens':usage.get('total_tokens',0),
    }
    answer_text=response.content

    try:
        json_response=json.loads(answer_text)
        response_df=pd.DataFrame([json_response])
    except json.JSONDecodeError:
        response_df=pd.DataFrame([{'answer':answer_text}])

    token_use_df=pd.DataFrame([token_dict])
    return response_df,token_use_df

def get_answer_from_deepseek_response(reponse):
    """从 invoke 返回的 AIMessage 提取回答文本。"""
    return reponse.content

In [ ]:
# Define a custom# Define a custom execution loop for RAG (适配本地 Milvus Lite + DeepSeek + ragas 0.4.3)
def process_user_message(user_input, message_history,top_k=3,debug=False):
    delimiter="```"
    threshold_retrieval_score=0.6

    # STEP 2: 本地 Milvus Lite 检索（替代原版 STEP 2 + STEP 4 的 Zilliz 双 collection 分支）
    if debug:
        print()
        print("STEP 2: Retrieval from local Milvus Lite collection (wikipedia).")
    contexts,hits=milvus_lite_collection_search(user_input,top_k=top_k)
    distance_score=hits[0]['distance']
    retrieval_context=hits[0]['entity']['chunk']
    retrieval_source=hits[0]['entity']['source']
    if debug:
        print(f"DISTNACE SCORE:{distance_score}")
        print(f"chunk_answer:{retrieval_context[:150]}")

    # STEP 3+4: 原版的"分数低→判断 intent→换 collection"逻辑
        # 由于只有一个 wikipedia collection，此分支简化为直接拒绝低分结果
    if distance_score<threshold_retrieval_score:
        print("UNABLE TO MATCH INTENT WITH ANY INTERNAL DOC STORE")
        return "Sorry, we cannot process this request.",message_history

    # STEP 5: DeepSeek 生成回答
    if debug:
        print()
        print("STEP 5: Generating answer with deepseek-v4-flash.")
    system_message=f"""
    Use the Context below to answer the user's question. Be clear, factual, complete, concise.
    If the answer is not in the Context, say "I don't know".  Otherwise answer using this format:
    Context: {retrieval_context}
    Answer: The answer to the question.
    Grounding source: {retrieval_source}
    """
    messages=[
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': f"{delimiter}{user_input}{delimiter}"},
        {'role': 'assistant', 'content': f"Relevant context:\n{retrieval_context}"}
    ]
    final_response = llm_langchain.invoke(message_history + messages)   # ChatDeepSeek
    message_history = message_history + messages[1:]
    answer = final_response.content

    # STEP 6: ragas 0.4.3 评估
    if debug:
        print()
        print("STEP 6: Evaluate the chatbot response with ragas.")
        # 注意：ds 需要预先构建（question/contexts/answer/ground_truth 四列）
        # 这里用单条记录示例
        from datasets import Dataset
        from ragas import evaluate
        from ragas.metrics import answer_similarity, answer_relevancy, answer_correctness
        ds = Dataset.from_dict({
            "question": [user_input],
            "contexts": [contexts],
            "answer": [answer],
            "ground_truth": ["<标准答案>"],   # 需要人工提供
        })
        ragas_result = evaluate(
            ds,
            metrics=[answer_similarity, answer_relevancy, answer_correctness],
            llm=llm_langchain,
            embeddings=lc_embeddings,        # 你的 HuggingFaceEmbeddings
        )
        ragas_df = ragas_result.to_pandas()
        print(f"Ragas: answer_similarity={ragas_df.answer_similarity[0]:.3f}, "
              f"answer_relevancy={ragas_df.answer_relevancy[0]:.3f}, "
              f"answer_correctness={ragas_df.answer_correctness[0]:.3f}")

    # STEP 7: 返回答案
    return answer, message_history

In [18]:
# 定义与 NYC 维基百科知识库匹配的测试问题
test_questions = [
    "What is the population of New York City?",
    "When was New York City founded?",
    "What are the five boroughs of New York City?",
    "What is the area of New York City?",
]

QUESTION_NUMBER = 3
SAMPLE_QUESTION = test_questions[QUESTION_NUMBER]
print(f"question = {SAMPLE_QUESTION}")

all_messages = []
answer_history = []
deepseek_answer, messages = process_user_message(SAMPLE_QUESTION, all_messages, debug=True)
answer_history.append(deepseek_answer)
pprint(f"Answer: {deepseek_answer}")

question = What is the area of New York City?

STEP 2: Retrieval from local Milvus Lite collection (wikipedia).
DISTNACE SCORE:0.7453761100769043
chunk_answer:area, the largest metropolitan area in the United States by both population and urban area. New York is a global center of technology,[10] finance[11]

STEP 5: Generating answer with deepseek-v4-flash.

STEP 6: Evaluate the chatbot response with ragas.


C:\Users\Administrator\AppData\Local\Temp\ipykernel_10040\2710838371.py:52: DeprecationWarning: Importing answer_similarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_similarity
  from ragas.metrics import answer_similarity, answer_relevancy, answer_correctness
C:\Users\Administrator\AppData\Local\Temp\ipykernel_10040\2710838371.py:52: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import answer_similarity, answer_relevancy, answer_correctness
C:\Users\Administrator\AppData\Local\Temp\ipykernel_10040\2710838371.py:52: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' inst

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Ragas: answer_similarity=0.507, answer_relevancy=0.000, answer_correctness=0.127
"Answer: I don't know."
